In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import torch
import yaml
from NN_models import Model18, Model20
from MPCModelFuncs import differential_mpc_model, differential_mpc_model_no_load
from MPCModelFuncs import euler_step

# Data directory
dir = '/home/david/fst/autonomous-systems/src/control/learning_mpcc/AI/train_data/estril_19_06_T1/'
model_id1 = 18
model_id2 = 20
modelName1 = 'model_18_estril_19_06_T1'
modelName2 = 'model_20_estril_19_06_T1'
T = 1
h = 0.050
dynamic_model = 0 # 0: No load trasnfer, 1: Load transfer
percentage_to_show = 0.1

x_test, y_test = [], []
# Load data
npz_files = [f for f in os.listdir(dir) if f.endswith(".npz")]
for file in npz_files:
    data = np.load(dir + file)
    x_test.append(data['x_test'])
    y_test.append(data['y_test'])
x_test = np.concatenate(x_test, axis=0)
y_test = np.concatenate(y_test, axis=0)

# Read yaml file
p = np.zeros(22)
with open('/home/david/fst/autonomous-systems/src/control/learning_mpcc/config/default.yaml') as stream:
    parameters = yaml.load(stream, Loader=yaml.SafeLoader)
    p[0] = parameters['model_params']['l_f'];
    p[1] = parameters['model_params']['l_r'];
    p[2] = parameters['model_params']['m'];
    p[3] = parameters['model_params']['I_z'];
    p[4] = parameters['model_params']['T_max_front'];
    p[5] = parameters['model_params']['T_max_rear'];
    p[6] = parameters['model_params']['T_brake_front'];
    p[7] = parameters['model_params']['T_brake_rear'];
    p[8] = parameters['model_params']['GR'];
    p[9] = parameters['model_params']['eta_motor'];
    p[10] = parameters['model_params']['r_wheel'];
    p[11] = parameters['model_params']['g'];
    p[12] = parameters['model_params']['C_roll'];
    p[13] = parameters['model_params']['rho'];
    p[14] = parameters['model_params']['C_d'];
    p[15] = parameters['model_params']['C_l'];
    p[16] = parameters['tyre_params']['B'];
    p[17] = parameters['tyre_params']['C'];
    p[18] = parameters['tyre_params']['D'];
    p[19] = parameters['model_params']['downforce_front'];
    p[20] = parameters['model_params']['downforce_rear'];
    p[21] = parameters['model_params']['h_cog'];

In [ ]:
if model_id1 == 18:
    model1 = Model18()
elif model_id1 == 20:
    model1 = Model20()
    
if model_id2 == 18:
    model2 = Model18()
elif model_id2 == 20:
    model2 = Model20()

state_dict1 = torch.load('../saved_weights/' + modelName1 + '.pth')
model1.load_state_dict(state_dict1)
model1.eval()

state_dict2 = torch.load('../saved_weights/' + modelName2 + '.pth')
model2.load_state_dict(state_dict2)
model2.eval()

In [ ]:
states = np.array([x_test[:,-1,0], x_test[:,-1,1], x_test[:,-1,2]]).T
true_NN_correction = np.array([y_test[:,0], y_test[:,1], y_test[:,2]]).T
control_actions = np.array([x_test[:,-1,0], x_test[:,-1,1], x_test[:,-2,0], x_test[:,-2,1]]).T
if dynamic_model == 0:
    dX = differential_mpc_model_no_load(states, control_actions, p).T
elif dynamic_model == 1:
    dX = differential_mpc_model(states, control_actions, p).T
x_test = torch.from_numpy(x_test).float()
NN_correction1 = model1(x_test)
NN_correction2 = model2(x_test)
states_next = euler_step(states, dX+true_NN_correction, h)
states_next_model = euler_step(states, dX, h)
states_next_NN1 = euler_step(states, dX+NN_correction1.detach().numpy(), h)
states_next_NN2 = euler_step(states, dX+NN_correction2.detach().numpy(), h)

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(50, 50), dpi=300)
size = len(states_next[:,0])

ax[0].plot(states_next[:int(percentage_to_show*size),0], label='True',linewidth=0.1)
ax[0].plot(states_next_model[:int(percentage_to_show*size),0], label='Model Predicted',linewidth=0.1, linestyle='--')
ax[0].plot(states_next_NN1[:int(percentage_to_show*size),0], label='NN Predicted 1',linewidth=0.1, linestyle=':')
ax[0].plot(states_next_NN2[:int(percentage_to_show*size),0], label='NN Predicted 2',linewidth=0.1, linestyle=':')
ax[0].set_xlabel('Timestep',fontsize=5)
ax[0].set_ylabel('Velocity x [m/s]',fontsize=5)
ax[0].legend(fontsize=2)

ax[1].plot(states_next[:int(percentage_to_show*size),1], label='True',linewidth=0.1)
ax[1].plot(states_next_model[:int(percentage_to_show*size),1], label='Model Predicted',linewidth=0.1, linestyle='--')
ax[1].plot(states_next_NN1[:int(percentage_to_show*size),1], label='NN Predicted 1',linewidth=0.1, linestyle=':')
ax[1].plot(states_next_NN2[:int(percentage_to_show*size),1], label='NN Predicted 2',linewidth=0.1, linestyle=':')
ax[1].set_xlabel('Timestep',fontsize=5)
ax[1].set_ylabel('Velocity y [m/s]',fontsize=5)
ax[1].legend(fontsize=2)

ax[2].plot(states_next[:int(percentage_to_show*size),2], label='True',linewidth=0.1)
ax[2].plot(states_next_model[:int(percentage_to_show*size),2], label='Model Predicted',linewidth=0.1, linestyle='--')
ax[2].plot(states_next_NN1[:int(percentage_to_show*size),2], label='NN Predicted 1',linewidth=0.1, linestyle=':')
ax[2].plot(states_next_NN2[:int(percentage_to_show*size),2], label='NN Predicted 2',linewidth=0.1, linestyle=':')
ax[2].set_xlabel('Timestep',fontsize=5)
ax[2].set_ylabel('Yaw Rate [rad/s]',fontsize=5)
ax[2].legend(fontsize=2)

plt.savefig("../saved_images/2NNs_comparison.pdf", format='pdf', bbox_inches='tight')
plt.show()

In [ ]:
RMSE_vx_model = np.sqrt(np.mean((states_next[:,0] - states_next_model[:,0])**2))
RMSE_vy_model = np.sqrt(np.mean((states_next[:,1] - states_next_model[:,1])**2))
RMSE_r_model = np.sqrt(np.mean((states_next[:,2]- states_next_model[:,2])**2))

RMSE_vx_NN1 = np.sqrt(np.mean((states_next[:,0] - states_next_NN1[:,0])**2))
RMSE_vy_NN1 = np.sqrt(np.mean((states_next[:,1] - states_next_NN1[:,1])**2))
RMSE_r_NN1 = np.sqrt(np.mean((states_next[:,2] - states_next_NN1[:,2])**2))

RMSE_vx_NN2 = np.sqrt(np.mean((states_next[:,0] - states_next_NN2[:,0])**2))
RMSE_vy_NN2 = np.sqrt(np.mean((states_next[:,1] - states_next_NN2[:,1])**2))
RMSE_r_NN2 = np.sqrt(np.mean((states_next[:,2] - states_next_NN2[:,2])**2))

max_vx_model = np.max(np.abs(states_next[:,0] - states_next_model[:,0]))
max_vy_model = np.max(np.abs(states_next[:,1] - states_next_model[:,1]))
max_r_model = np.max(np.abs(states_next[:,2] - states_next_model[:,2]))

max_vx_NN1 = np.max(np.abs(states_next[:,0] - states_next_NN1[:,0]))
max_vy_NN1 = np.max(np.abs(states_next[:,1] - states_next_NN1[:,1]))
max_r_NN1 = np.max(np.abs(states_next[:,2] - states_next_NN1[:,2]))

max_vx_NN2 = np.max(np.abs(states_next[:,0] - states_next_NN2[:,0]))
max_vy_NN2 = np.max(np.abs(states_next[:,1] - states_next_NN2[:,1]))
max_r_NN2 = np.max(np.abs(states_next[:,2] - states_next_NN2[:,2]))

print('RMSE of the true and predicted states:')
print("Model  - vx: {:.4f}, vy: {:.4f}, r: {:.4f}".format(RMSE_vx_model, RMSE_vy_model, RMSE_r_model))
print("NN1    - vx: {:.4f}, vy: {:.4f}, r: {:.4f}".format(RMSE_vx_NN1, RMSE_vy_NN1, RMSE_r_NN1))
print("NN2    - vx: {:.4f}, vy: {:.4f}, r: {:.4f}".format(RMSE_vx_NN2, RMSE_vy_NN2, RMSE_r_NN2))
print('Highest Error in:')
print("Model  - vx: {:.4f}, vy: {:.4f}, r: {:.4f}".format(max_vx_model, max_vy_model, max_r_model))
print("NN1    - vx: {:.4f}, vy: {:.4f}, r: {:.4f}".format(max_vx_NN1, max_vy_NN1, max_r_NN1))
print("NN2    - vx: {:.4f}, vy: {:.4f}, r: {:.4f}".format(max_vx_NN2, max_vy_NN2, max_r_NN2))